In [10]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new(target portfolio ).csv',
    encoding='latin-1'
)

df['company_normalized'] = df['company'].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
pivot_data = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for year in years:
        year_data = company_data[company_data['year'] == year]
        if not year_data.empty:
            row[f'region_{year}'] = year_data['region'].iloc[0]
            row[f'sector_{year}'] = year_data['sector'].iloc[0]
            row[f're100_{year}'] = year_data['re100'].iloc[0]
            row[f'sbti_{year}'] = year_data['sbti'].iloc[0]
            row[f'cn_{year}'] = year_data['cn'].iloc[0]
            row[f'nz_{year}'] = year_data['nz'].iloc[0]
            row[f'cc_{year}'] = year_data['cc'].iloc[0]
    
    pivot_data.append(row)

matrix = pd.DataFrame(pivot_data)
matrix.to_csv('company_matrix.csv', index=False)

KeyError: 'company'

In [ ]:

# Create pivot: one row per company with columns for each year-target combination
years = sorted(df['year'].unique())
pivot_data = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for year in years:
        year_data = company_data[company_data['year'] == year]
        if not year_data.empty:
            row[f'region_{year}'] = year_data['region'].iloc[0]
            row[f'sector_{year}'] = year_data['sector'].iloc[0]
            row[f're100_{year}'] = year_data['re100'].iloc[0]
            row[f'sbti_{year}'] = year_data['sbti'].iloc[0]
            row[f'cn_{year}'] = year_data['cn'].iloc[0]
            row[f'nz_{year}'] = year_data['nz'].iloc[0]
            row[f'cc_{year}'] = year_data['cc'].iloc[0]
    
    pivot_data.append(row)

matrix = pd.DataFrame(pivot_data)

# Track CN 2021 -> NZ 2025
cn_to_nz = matrix[
    (matrix['cn_2021'] == 1) & 
    (matrix['nz_2025'] == 1)
]['company'].tolist()

# Track SBTi changes every two years
sbti_transitions = {}
for i in range(len(years) - 1):
    year1, year2 = years[i], years[i+1]
    if year2 - year1 <= 2:
        gained = matrix[
            (matrix[f'sbti_{year1}'] != 1) & 
            (matrix[f'sbti_{year2}'] == 1)
        ]['company'].tolist()
        lost = matrix[
            (matrix[f'sbti_{year1}'] == 1) & 
            (matrix[f'sbti_{year2}'] != 1)
        ]['company'].tolist()
        sbti_transitions[f'{year1}_to_{year2}'] = {'gained': gained, 'lost': lost}

print(f"Companies with CN in 2021 and NZ in 2025: {len(cn_to_nz)}")
print(f"\nSBTi transitions: {sbti_transitions}")

matrix.to_csv('company_matrix.csv', index=False)